<a href="https://colab.research.google.com/github/CPTR295/NLP-Basic-Using-Pytorch/blob/main/Document_Classification_AG_News.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import collections
import numpy as np
import pandas as pd
import re

from argparse import Namespace

In [2]:
!kaggle datasets download -d amananandrai/ag-news-classification-dataset


Dataset URL: https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset
License(s): unknown
100% 11.4M/11.4M [00:01<00:00, 7.01MB/s]



In [3]:
!unzip ag-news-classification-dataset.zip

Archive:  ag-news-classification-dataset.zip
  inflating: test.csv                
  inflating: train.csv               


In [19]:
df1 = pd.read_csv('/content/train.csv')
df2 = pd.read_csv('/content/test.csv')
df=pd.concat([df1, df2], axis=0)
df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [20]:
df.columns

Index(['Class Index', 'Title', 'Description'], dtype='object')

In [21]:
df['Class Index'].value_counts()

,count
Class Index,
3,31900
4,31900
2,31900
1,31900


In [22]:
df['category']=df['Class Index']

In [24]:
args = Namespace(
    raw_dataset_csv="/content/train.csv",
    train_proportion=0.7,
    val_proportion=0.15,
    test_proportion=0.15,
    output_munged_csv="/content/news_with_splits.csv",
    seed=1337
)

In [23]:
by_category = collections.defaultdict(list)
for _, row in df.iterrows():
    by_category[row.category].append(row.to_dict())

In [25]:
# Create split data
final_list = []
np.random.seed(args.seed)
for _, item_list in sorted(by_category.items()):
    np.random.shuffle(item_list)
    n = len(item_list)
    n_train = int(args.train_proportion*n)
    n_val = int(args.val_proportion*n)
    n_test = int(args.test_proportion*n)

    # Give data point a split attribute
    for item in item_list[:n_train]:
        item['split'] = 'train'
    for item in item_list[n_train:n_train+n_val]:
        item['split'] = 'val'
    for item in item_list[n_train+n_val:]:
        item['split'] = 'test'

    # Add to final list
    final_list.extend(item_list)

In [26]:
final_news = pd.DataFrame(final_list)

In [27]:
final_news.split.value_counts()

,count
split,
train,89320
val,19140
test,19140


In [29]:
# Preprocess the reviews
def preprocess_text(text):
    text = ' '.join(word.lower() for word in text.split(" "))
    text = re.sub(r"([.,!?])", r" \1 ", text)
    text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
    return text

final_news.title = final_news.Title.apply(preprocess_text)

/tmp/ipykernel_1019/3324291000.py:8: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  final_news.title = final_news.Title.apply(preprocess_text)


In [30]:
final_news.head()

,Class Index,Title,Description,category,split
0,1,Shaukat Aziz gets vote of confidence,ISLAMABAD: Newly-elected known as finance wiza...,1,train
1,1,The Poetry of 'Gimme an A! Gimme an S! Gimme a...,"Jonny Hurst, Britain's soccer chant laureate, ...",1,train
2,1,US Study Says a Nuclear Iran Would Aid More Te...,Iran could acquire a nuclear bomb in the next ...,1,train
3,1,Israeli parliament set for historic vote on Ga...,JERUSALEM : The Israeli parliament is set for ...,1,train
4,1,India #39;s decision on troop reduction cosmet...,"Islamabad, Nov. 20. (PTI): Pakistan President ...",1,train


In [31]:
final_news.to_csv(args.output_munged_csv, index=False)

In [32]:
import os
from argparse import Namespace
from collections import Counter
import json
import re
import string

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm_notebook

In [62]:
class Vocabulary(object):
  def __init__(self,token_to_idx=None):
    if token_to_idx is None:
      token_to_idx = {}
    self._token_to_idx = token_to_idx
    self._idx_to_token = {idx:token for token,idx in self._token_to_idx.items()}


  def to_serializable(self):
    return {'token_to_idx':self._token_to_idx}

  @classmethod
  def from_serializable(cls,contents):
    return cls(**contents)

  def add_token(self,token):
    if token in self._token_to_idx:
      index = self._token_to_idx[token]
    else:
      index = len(self._token_to_idx)
      self._token_to_idx[token] = index
      self._idx_to_token[index] = token
    return index

  def add_many(self,tokens):
    return [self.add_token(token) for token in tokens]

  def lookup_token(self,token):
    return self._token_to_idx[token]

  def lookup_index(self,index):
    if index not in self._idx_to_token:
      raise KeyError('The index not in vocab')
    return self._idx_to_token[index]

  def __str__(self):
    return f'<Vocabulary(size={len(self)})>'

  def __len__(self):
    return len(self._token_to_idx)

  def __str__(self) -> str:
    return f'<Vocabulary(size={len(self)})>'

  def __len__(self) -> int:
    return len(self._token_to_idx)

In [63]:
class SequenceVocabulary(Vocabulary):
  def __init__(self, token_to_idx=None, unk_token="<UNK>",
                 mask_token="<MASK>", begin_seq_token="<BEGIN>",
                 end_seq_token="<END>"):
    super(SequenceVocabulary,self).__init__(token_to_idx)
    self._mask_token = mask_token
    self._unk_token = unk_token
    self._begin_seq_token = begin_seq_token
    self._end_seq_token = end_seq_token

    self.mask_index = self.add_token(self._mask_token)
    self.unk_index = self.add_token(self._unk_token)
    self.begin_seq_index = self.add_token(self._begin_seq_token)
    self.end_seq_index = self.add_token(self._end_seq_token)

  def to_serializable(self):
    contents = super(SequenceVocabulary,self).to_serializable()
    contents.update({'unk_token':self._unk_token,
                    'mask_token':self._mask_token,
                    'begin_seq_token':self._begin_seq_token,
                    'end_seq_token':self._end_seq_token})
    return contents

  def lookup_token(self,token):
    if self.unk_index >= 0:
      return self._token_to_idx.get(token,self.unk_index)
    else:
      return self._token_to_idx[token]

In [84]:
class NewsVectorizer(object):
  def __init__(self,title_vocab,category_vocab):
    self.title_vocab = title_vocab
    self.category_vocab = category_vocab

  def vectorize(self,title,vector_length=-1):
    indices = [self.title_vocab.begin_seq_index]
    indices.extend(self.title_vocab.lookup_token(token) for token in title.split(' '))
    indices.append(self.title_vocab.end_seq_index)

    # if vector_length < 0:
    #   vector_length = len(indices)
    vector_length = max(vector_length, len(indices))
    out_vector = np.zeros(vector_length,dtype=np.int64)
    out_vector[:len(indices)] = indices
    out_vector[len(indices):] = self.title_vocab.mask_index
    return out_vector

  @classmethod
  def from_dataframe(cls,news_df,cutoff=25):
    title_vocab = SequenceVocabulary()
    category_vocab = Vocabulary()
    for category in sorted(set(news_df.category)):
      category_vocab.add_token(category)
    word_counts = Counter()
    for title in news_df.Title:
      for token in title.split(' '):
        if token not in string.punctuation:
          word_counts[token] += 1
    for word,count in word_counts.items():
      if count > cutoff:
        title_vocab.add_token(word)
    return cls(title_vocab,category_vocab)

  @classmethod
  def from_serializable(cls,contents):
    title_vocab = SequenceVocabulary.from_serializable(contents['title_vocab'])
    category_vocab = Vocabulary.from_serializable(contents['category_vocab'])
    return cls(title_vocab=title_vocab,category_vocab=category_vocab)

  def to_serializable(self):
    return {'title_vocab':self.title_vocab.to_serializable(),
            'category_vocab':self.category_vocab.to_serializable()}


In [92]:
class NewsDataset(Dataset):
  def __init__(self,news_df,vectorizer):
    self.news_df = news_df
    self._vectorizer = vectorizer

    measure_len = lambda context: len(context.split(" "))
    self._max_seq_length = max(map(measure_len, news_df.Title))+2

    self.train_df = self.news_df[self.news_df.split=='train']
    self.train_size = len(self.train_df)

    self.val_df = self.news_df[self.news_df.split=='val']
    self.validation_size = len(self.val_df)

    self.test_df = self.news_df[self.news_df.split=='test']
    self.test_size = len(self.test_df)

    self._lookup_dict = {'train': (self.train_df, self.train_size),
                          'val': (self.val_df, self.validation_size),
                          'test': (self.test_df, self.test_size)}

    self.set_split('train')
    class_counts = news_df.category.value_counts().to_dict()
    def sort_key(item):
        return self._vectorizer.category_vocab.lookup_token(item[0])
    sorted_counts = sorted(class_counts.items(), key=sort_key)
    frequencies = [count for _, count in sorted_counts]
    self.class_weights = 1.0 / torch.tensor(frequencies, dtype=torch.float32)

  @classmethod
  def load_dataset_and_make_vectorizer(cls,news_csv):
    news_df = pd.read_csv(news_csv)
    train_news_df = news_df[news_df.split=='train']
    return cls(news_df,NewsVectorizer.from_dataframe(train_news_df))

  @classmethod
  def load_dataset_and_load_vectorizer(cls,news_csv,vectorizer_filepath):
    news_df = pd.read_csv(news_csv)
    vectorizer = cls.load_vectorizer_only(vectorizer_filepath)
    return cls(news_csv,vectorizer)

  @classmethod
  def load_vectorizer_only(cls,vectorizer_filepath):
    with open(vectorizer_filepath) as fp:
      return NewsVectorizer.from_serializable(json.load(fp))

  def save_vectorizer(self,vectorizer_filepath):
    with open(vectorizer_filepath,'w') as fp:
      json.dump(self._vectorizer.to_serializable(),fp)

  def get_vectorizer(self):
    return self._vectorizer

  def set_split(self,split='train'):
    self._target_split = split
    self._target_df, self._target_size = self._lookup_dict[split]

  def __len__(self):
    return self._target_size

  def __getitem__(self,index):
    row = self._target_df.iloc[index]
    title_vector = self._vectorizer.vectorize(row.Title,self._max_seq_length)
    category_index = self._vectorizer.category_vocab.lookup_token(row.category)
    return {'x_data':title_vector,'y_target':category_index}

  def get_num_batches(self,batch_size):
    return len(self) // batch_size

In [39]:
def generate_batches(dataset,batch_size,shuffle=True,drop_last=True,device='cpu'):
  dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last)
  for data_dict in dataloader:
    out_data_dict = {}
    for name,tensor in data_dict.items():
      out_data_dict[name]=data_dict[name].to(device)
    yield out_data_dict

In [40]:
class NewsClassifier(nn.Module):
  def __init__(self,embedding_size,num_embeddings,num_channels,hidden_dim,num_classes,dropout_p,pretrained_embeddings=None,padding_idx=0):
    super(NewsClassifier, self).__init__()
    if pretrained_embeddings is None:
      self.emb = nn.Embedding(embedding_dim=embedding_size,num_embeddings=num_embeddings,padding_idx=padding_idx)
    else:
      pretrained_embeddings = torch.from_numpy(pretrained_embeddings).float()
      self.emb = nn.Embedding(embedding_dim=embedding_size,
                              num_embeddings=num_embeddings,
                              padding_idx=padding_idx,
                              _weight=pretrained_embeddings)
    self.convent = nn.Sequential(
        nn.Conv1d(in_channels=embedding_size,out_channels=num_channels,kernel_size=3),
        nn.ELU(),
        nn.Conv1d(in_channels=num_channels,out_channels=num_channels,kernel_size=3,stride=2),
        nn.ELU(),
        nn.Conv1d(in_channels=num_channels,out_channels=num_channels,kernel_size=3,stride=2),
        nn.ELU(),
        nn.Conv1d(in_channels=num_channels,out_channels=num_channels,kernel_size=3),
        nn.ELU()
    )

    self._dropout_p = dropout_p
    self.fc1 = nn.Linear(num_channels,hidden_dim)
    self.fc2 = nn.Linear(hidden_dim,num_classes)
  def forward(self,x_in,apply_softmax=False):
    x_embedded = self.emb(x_in).permute(0, 2, 1)
    features = self.convent(x_embedded)
    remaining_size = features.size(dim=2)
    features = F.avg_pool1d(features, remaining_size).squeeze(dim=2)
    features = F.dropout(features, p=self._dropout_p)
    intermediate_vector = F.relu(F.dropout(self.fc1(features), p=self._dropout_p))
    prediction_vector = self.fc2(intermediate_vector)

    if apply_softmax:
        prediction_vector = F.softmax(prediction_vector, dim=1)

    return prediction_vector

In [41]:
def make_train_state(args):
    return {'stop_early': False,
            'early_stopping_step': 0,
            'early_stopping_best_val': 1e8,
            'learning_rate': args.learning_rate,
            'epoch_index': 0,
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'test_loss': -1,
            'test_acc': -1,
            'model_filename': args.model_state_file}

In [42]:
def update_train_state(args, model, train_state):

    # Save model after first epoch
    if train_state["epoch_index"] == 0:
        torch.save(model.state_dict(), train_state["model_filename"])

        train_state["stop_early"] = False
        train_state["early_stopping_step"] = 0
        train_state["early_stopping_best_val"] = train_state["val_loss"][-1]

    else:
        current_val_loss = train_state["val_loss"][-1]

        # Validation loss improved
        if current_val_loss < train_state["early_stopping_best_val"]:

            train_state["early_stopping_best_val"] = current_val_loss
            train_state["early_stopping_step"] = 0

            # Save best model
            torch.save(model.state_dict(), train_state["model_filename"])

        # Validation loss did not improve
        else:
            train_state["early_stopping_step"] += 1

        train_state["stop_early"] = (
            train_state["early_stopping_step"] >=
            args.early_stopping_criteria
        )

    return train_state

In [43]:
def compute_accuracy(y_pred, y_target):
    _, y_pred_indices = y_pred.max(dim=1)
    n_correct = torch.eq(y_pred_indices, y_target).sum().item()
    return n_correct / len(y_pred_indices) * 100

In [44]:
def set_seed_everywhere(seed, cuda):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if cuda:
        torch.cuda.manual_seed_all(seed)

def handle_dirs(dirpath):
    if not os.path.exists(dirpath):
        os.makedirs(dirpath)

In [79]:
with open('/content/glove.6B.100d.txt', "r") as f:
  f.readline()
  print(f.readline())

, -0.10767 0.11053 0.59812 -0.54361 0.67396 0.10663 0.038867 0.35481 0.06351 -0.094189 0.15786 -0.81665 0.14172 0.21939 0.58505 -0.52158 0.22783 -0.16642 -0.68228 0.3587 0.42568 0.19021 0.91963 0.57555 0.46185 0.42363 -0.095399 -0.42749 -0.16567 -0.056842 -0.29595 0.26037 -0.26606 -0.070404 -0.27662 0.15821 0.69825 0.43081 0.27952 -0.45437 -0.33801 -0.58184 0.22364 -0.5778 -0.26862 -0.20425 0.56394 -0.58524 -0.14365 -0.64218 0.0054697 -0.35248 0.16162 1.1796 -0.47674 -2.7553 -0.1321 -0.047729 1.0655 1.1034 -0.2208 0.18669 0.13177 0.15117 0.7131 -0.35215 0.91348 0.61783 0.70992 0.23955 -0.14571 -0.37859 -0.045959 -0.47368 0.2385 0.20536 -0.18996 0.32507 -1.1112 -0.36341 0.98679 -0.084776 -0.54008 0.11726 -1.0194 -0.24424 0.12771 0.013884 0.080374 -0.35414 0.34951 -0.7226 0.37549 0.4441 -0.99059 0.61214 -0.35111 -0.83155 0.45293 0.082577



In [80]:
def load_glove_from_file(glove_filepath):
    word_to_index = {}
    embeddings =[]
    with open(glove_filepath, "r") as f:
        for index,line in enumerate(f):
            line = line.split(" ")# each line: word num1 num2 ...
            word_to_index[line[0]] = index
            embedding_i = np.array([float(val) for val in line[1:]])
            embeddings.append(embedding_i)
    return word_to_index, np.stack(embeddings)

def make_embedding_matrix(glove_filepath,words):
  word_to_idx, glove_embeddings = load_glove_from_file(glove_filepath)
  embedding_size = glove_embeddings.shape[1]
  final_embeddings = np.zeros((len(words), embedding_size))

  for i, word in enumerate(words):
      if word in word_to_idx:
          final_embeddings[i, :] = glove_embeddings[word_to_idx[word]]
      else:
          embedding_i = torch.ones(1, embedding_size)
          torch.nn.init.xavier_uniform_(embedding_i)
          final_embeddings[i, :] = embedding_i

  return final_embeddings

In [46]:
from argparse import Namespace

In [54]:
args = Namespace(
    # Data and Path hyper parameters
    news_csv="/content/news_with_splits.csv",
    vectorizer_file="vectorizer.json",
    model_state_file="model.pth",
    save_dir="/content/",
    # Model hyper parameters
    glove_filepath='/content/glove.6B.100d.txt',
    use_glove=False,
    embedding_size=100,
    hidden_dim=100,
    num_channels=100,
    # Training hyper parameter
    seed=1337,
    learning_rate=0.001,
    dropout_p=0.1,
    batch_size=128,
    num_epochs=100,
    early_stopping_criteria=5,
    # Runtime option
    cuda=True,
    catch_keyboard_interrupt=True,
    reload_from_files=False,
    expand_filepaths_to_save_dir=True
)

In [49]:
if args.expand_filepaths_to_save_dir:
    args.vectorizer_file = os.path.join(args.save_dir,
                                        args.vectorizer_file)

    args.model_state_file = os.path.join(args.save_dir,
                                         args.model_state_file)

    print("Expanded filepaths: ")
    print("\t{}".format(args.vectorizer_file))
    print("\t{}".format(args.model_state_file))

Expanded filepaths: 
	/content/vectorizer.json
	/content/model.pth


In [74]:
# Check CUDA
if not torch.cuda.is_available():
    args.cuda = False

args.device = torch.device("cuda" if args.cuda else "cpu")
print("Using CUDA: {}".format(args.cuda))
args.device

Using CUDA: True


device(type='cuda')

In [51]:
set_seed_everywhere(args.seed, args.cuda)
handle_dirs(args.save_dir)

In [53]:
!wget https://nlp.stanford.edu/data/glove.6B.zip
#Stanford Embedding
!unzip glove.6B.zip

--2026-07-12 10:45:53--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-07-12 10:45:53--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.02MB/s    in 2m 41s  

2026-07-12 10:48:35 (5.12 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflatin

In [71]:
args.use_glove = True

In [93]:
if args.reload_from_files:
    print("Loading dataset and loading vectorizer")
    dataset = NewsDataset.load_dataset_and_load_vectorizer(args.news_csv,
                                                           args.vectorizer_file)
else:
    print("Loading dataset and creating vectorizer")
    dataset = NewsDataset.load_dataset_and_make_vectorizer(args.news_csv)
    dataset.save_vectorizer(args.vectorizer_file)

Loading dataset and creating vectorizer


In [94]:
vectorizer = dataset.get_vectorizer()

In [95]:
if args.use_glove:
    words = vectorizer.title_vocab._token_to_idx.keys()
    embeddings = make_embedding_matrix(glove_filepath=args.glove_filepath,words=words)
    print("Using pre-trained embeddings")
else:
    print("Not using pre-trained embeddings")
    embeddings = None

Using pre-trained embeddings


/tmp/ipykernel_1019/734709806.py:23: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  final_embeddings[i, :] = embedding_i


In [96]:
classifier = NewsClassifier(embedding_size=args.embedding_size,
                            num_embeddings=len(vectorizer.title_vocab),
                            num_channels=args.num_channels,
                            hidden_dim=args.hidden_dim,
                            num_classes=len(vectorizer.category_vocab),
                            dropout_p=args.dropout_p,
                            pretrained_embeddings=embeddings,
                            padding_idx=0)

In [97]:
classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)

loss_func = nn.CrossEntropyLoss(dataset.class_weights)
optimizer = optim.Adam(classifier.parameters(), lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,
                                           mode='min', factor=0.5,
                                           patience=1)

train_state = make_train_state(args)

In [98]:
try:
  for epoch_index in range(args.num_epochs):
    dataset.set_split('train')
    classifier.train()
    batch_generator = generate_batches(dataset,batch_size=args.batch_size,device=args.device)
    loss=0.0
    acc=0.0
    for batch_index,batch_dict in enumerate(batch_generator):
      optimizer.zero_grad()
      y_pred = classifier(x_in=batch_dict['x_data'])
      loss = loss_func(y_pred,batch_dict['y_target'])
      loss_batch = loss.item()
      loss.backward()
      optimizer.step()
      acc_batch = compute_accuracy(y_pred,batch_dict['y_target'])
      loss+=(loss_batch-loss)/(batch_index+1)
      acc+=(acc_batch-acc)/(batch_index+1)
    train_state['train_loss'].append(loss)
    train_state['train_acc'].append(acc)

    dataset.set_split('val')
    batch_generator = generate_batches(dataset,batch_size=args.batch_size,device=args.device)
    loss=0.0
    acc=0.0
    classifier.eval()
    with torch.no_grad():
      for batch_index,batch_dict in enumerate(batch_generator):
        y_pred = classifier(x_in=batch_dict['x_data'])
        loss = loss_func(y_pred,batch_dict['y_target'])
        loss_batch = loss.item()
        acc_batch = compute_accuracy(y_pred,batch_dict['y_target'])
        loss+=(loss_batch-loss)/(batch_index+1)
        acc+=(acc_batch-acc)/(batch_index+1)
      train_state['val_acc'].append(acc)
      train_state['val_loss'].append(loss)

    train_state = update_train_state(args=args,model=classifier,train_state=train_state)
    scheduler.step(train_state['val_loss'][-1])
    if train_state['stop_early']:
      break
    print("Epoch: {}/{}...".format(epoch_index+1,args.num_epochs),
          "Train Loss: {:.3f}".format(train_state['train_loss'][-1]),
          "Train Acc: {:.2f}".format(train_state['train_acc'][-1]),
          "Val Loss: {:.3f}".format(train_state['val_loss'][-1]),
          "Val Acc: {:.2f}".format(train_state['val_acc'][-1]))


except KeyboardInterrupt:
  print('Exiting')


Epoch: 1/100... Train Loss: 0.716 Train Acc: 71.07 Val Loss: 0.671 Val Acc: 78.52
Epoch: 2/100... Train Loss: 0.516 Train Acc: 80.40 Val Loss: 0.484 Val Acc: 80.07
Epoch: 3/100... Train Loss: 0.448 Train Acc: 81.44 Val Loss: 0.537 Val Acc: 79.85
Epoch: 4/100... Train Loss: 0.455 Train Acc: 82.15 Val Loss: 0.502 Val Acc: 79.69
Epoch: 5/100... Train Loss: 0.461 Train Acc: 83.44 Val Loss: 0.580 Val Acc: 80.46
Epoch: 6/100... Train Loss: 0.656 Train Acc: 83.86 Val Loss: 0.553 Val Acc: 80.11
Epoch: 7/100... Train Loss: 0.581 Train Acc: 84.78 Val Loss: 0.442 Val Acc: 80.35
Epoch: 8/100... Train Loss: 0.342 Train Acc: 84.99 Val Loss: 0.720 Val Acc: 79.97
Epoch: 9/100... Train Loss: 0.490 Train Acc: 85.35 Val Loss: 0.656 Val Acc: 80.16
Epoch: 10/100... Train Loss: 0.409 Train Acc: 86.05 Val Loss: 0.623 Val Acc: 80.08
Epoch: 11/100... Train Loss: 0.341 Train Acc: 86.21 Val Loss: 0.542 Val Acc: 80.12
Epoch: 12/100... Train Loss: 0.286 Train Acc: 86.55 Val Loss: 0.650 Val Acc: 79.87
Epoch: 13/100

In [99]:
dataset.set_split('test')
batch_generator = generate_batches(dataset,
                                   batch_size=args.batch_size,
                                   device=args.device)
running_loss = 0.0
running_acc = 0.0
classifier.eval()
with torch.no_grad():
    for batch_index,batch_dict in enumerate(batch_generator):
      y_pred = classifier(x_in=batch_dict['x_data'])
      loss = loss_func(y_pred,batch_dict['y_target'])
      loss_batch = loss.item()
      acc_batch = compute_accuracy(y_pred,batch_dict['y_target'])
      loss+=(loss_batch-loss)/(batch_index+1)
      acc+=(acc_batch-acc)/(batch_index+1)
train_state['test_loss'] = loss
train_state['test_acc'] = acc
print("Test loss: {};".format(train_state['test_loss']))
print("Test Accuracy: {}".format(train_state['test_acc']))

Test loss: 0.6240560412406921;
Test Accuracy: 79.44630872483222
